<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/01_Dataset_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
# ============================================================
# CELL 01.1 — DATASET ACQUISITION
# ============================================================
#
# Purpose:
#   Mount Google Drive and verify the AIR-LLM research project.
#
# Important:
#   Notebook 00 is the single source of truth for the project
#   configuration. This notebook does not redefine datasets,
#   targets, or experiment settings.
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"AIR-LLM project root not found:\n{PROJECT_ROOT}"
    )

print("=" * 100)
print("AIR-LLM — NOTEBOOK 01")
print("=" * 100)

print(f"Project root verified:\n{PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
AIR-LLM — NOTEBOOK 01
Project root verified:
/content/drive/MyDrive/AIR_LLM_Research


In [31]:
# ============================================================
# CELL 01.2 — RAW FILE DETECTION
# ============================================================

import os
import json
import yaml
import pandas as pd
import numpy as np

CONFIG_DIR = PROJECT_ROOT / "config"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"

for directory in [
    CONFIG_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    REFERENCE_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Load Notebook 00 configuration
# ------------------------------------------------------------

MASTER_CONFIG_PATH = (
    CONFIG_DIR / "config.yaml"
)

DATASET_REGISTRY_PATH = (
    CONFIG_DIR / "dataset_registry.yaml"
)

FEATURE_TARGET_REGISTRY_PATH = (
    CONFIG_DIR / "feature_target_registry.yaml"
)

for path in [
    MASTER_CONFIG_PATH,
    DATASET_REGISTRY_PATH,
    FEATURE_TARGET_REGISTRY_PATH
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required Notebook 00 configuration file not found:\n{path}"
        )


with open(
    MASTER_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:

    MASTER_CONFIG = yaml.safe_load(file)


with open(
    DATASET_REGISTRY_PATH,
    "r",
    encoding="utf-8"
) as file:

    DATASET_REGISTRY = yaml.safe_load(file)


with open(
    FEATURE_TARGET_REGISTRY_PATH,
    "r",
    encoding="utf-8"
) as file:

    FEATURE_TARGET_REGISTRY = yaml.safe_load(file)


# ------------------------------------------------------------
# Resolve dataset registry
# ------------------------------------------------------------

def resolve_registry_datasets(registry):

    if isinstance(registry, dict):

        if "datasets" in registry:
            datasets = registry["datasets"]

        elif "dataset_registry" in registry:
            datasets = registry["dataset_registry"]

        else:
            datasets = registry

    else:
        raise TypeError(
            "Dataset registry must be a dictionary."
        )

    return datasets


DATASETS_CONFIG = resolve_registry_datasets(
    DATASET_REGISTRY
)


EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]


missing_dataset_registry = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in DATASETS_CONFIG
]

if missing_dataset_registry:

    raise KeyError(
        "The following required datasets are missing "
        f"from dataset_registry.yaml: {missing_dataset_registry}"
    )


# ------------------------------------------------------------
# Detect raw files
# ------------------------------------------------------------

SUPPORTED_RAW_EXTENSIONS = {
    ".csv",
    ".txt",
    ".data",
    ".tsv"
}


ALL_RAW_FILES = sorted(
    [
        path
        for path in RAW_DIR.rglob("*")
        if path.is_file()
        and path.suffix.lower()
        in SUPPORTED_RAW_EXTENSIONS
    ]
)


def registry_value(dataset_config, keys, default=None):

    if not isinstance(dataset_config, dict):
        return default

    for key in keys:

        if key in dataset_config:
            value = dataset_config[key]

            if value is not None:
                return value

    return default


RAW_FILE_MAP = {}

for dataset_id in EXPECTED_DATASETS:

    dataset_config = DATASETS_CONFIG[
        dataset_id
    ]

    configured_path = registry_value(
        dataset_config,
        [
            "raw_path",
            "raw_file",
            "file_path",
            "source_path",
            "path"
        ]
    )

    configured_name = registry_value(
        dataset_config,
        [
            "raw_filename",
            "raw_file_name",
            "filename",
            "file_name"
        ]
    )

    candidate = None

    if configured_path:

        candidate_path = Path(
            str(configured_path)
        )

        if candidate_path.exists():
            candidate = candidate_path

        else:

            project_candidate = (
                PROJECT_ROOT
                / str(configured_path)
            )

            if project_candidate.exists():
                candidate = project_candidate

    if candidate is None and configured_name:

        matches = [
            path
            for path in ALL_RAW_FILES
            if path.name.lower()
            == str(configured_name).lower()
        ]

        if len(matches) == 1:
            candidate = matches[0]

    if candidate is None:

        dataset_tokens = [
            token.lower()
            for token in dataset_id.split("_")
        ]

        possible = [
            path
            for path in ALL_RAW_FILES
            if all(
                token in path.name.lower()
                for token in dataset_tokens
            )
        ]

        if len(possible) == 1:
            candidate = possible[0]

    RAW_FILE_MAP[dataset_id] = candidate


# ------------------------------------------------------------
# Detection report
# ------------------------------------------------------------

RAW_DETECTION_ROWS = []

for dataset_id in EXPECTED_DATASETS:

    path = RAW_FILE_MAP[dataset_id]

    RAW_DETECTION_ROWS.append(
        {
            "dataset_id": dataset_id,
            "raw_file_found": path is not None,
            "raw_path": str(path) if path else None,
            "file_name": path.name if path else None,
            "extension": path.suffix.lower()
            if path else None,
            "file_size_bytes": path.stat().st_size
            if path else None
        }
    )


RAW_DETECTION_DF = pd.DataFrame(
    RAW_DETECTION_ROWS
)

display(RAW_DETECTION_DF)


if RAW_DETECTION_DF["raw_file_found"].sum() != len(
    EXPECTED_DATASETS
):

    missing = RAW_DETECTION_DF.loc[
        ~RAW_DETECTION_DF["raw_file_found"],
        "dataset_id"
    ].tolist()

    raise FileNotFoundError(
        "Raw files not found for: "
        f"{missing}\n\n"
        f"Place the original raw files inside:\n{RAW_DIR}"
    )


print(
    f"\nDetected {len(EXPECTED_DATASETS)} / "
    f"{len(EXPECTED_DATASETS)} required raw datasets."
)

,dataset_id,raw_file_found,raw_path,file_name,extension,file_size_bytes
0,adult_income,True,/content/drive/MyDrive/AIR_LLM_Research/data/r...,adult_income.csv,.csv,3974305
1,bank_marketing,True,/content/drive/MyDrive/AIR_LLM_Research/data/r...,bank_marketing.csv,.csv,4610348
2,diabetes_130us,True,/content/drive/MyDrive/AIR_LLM_Research/data/r...,diabetes_130us.csv,.csv,16246860



Detected 3 / 3 required raw datasets.


In [32]:
# ============================================================
# CELL 01.3 — FILE FORMAT VALIDATION
# ============================================================

def detect_file_format(path):

    extension = path.suffix.lower()

    if extension == ".csv":
        return "csv"

    if extension == ".tsv":
        return "tsv"

    if extension in {".txt", ".data"}:
        return "delimited_text"

    return "unsupported"


FORMAT_ROWS = []

for dataset_id, path in RAW_FILE_MAP.items():

    file_format = detect_file_format(path)

    FORMAT_ROWS.append(
        {
            "dataset_id": dataset_id,
            "file_name": path.name,
            "extension": path.suffix.lower(),
            "detected_format": file_format,
            "supported": file_format != "unsupported",
            "size_bytes": path.stat().st_size
        }
    )


FILE_FORMAT_DF = pd.DataFrame(
    FORMAT_ROWS
)

display(FILE_FORMAT_DF)


if not FILE_FORMAT_DF["supported"].all():

    invalid = FILE_FORMAT_DF.loc[
        ~FILE_FORMAT_DF["supported"],
        "dataset_id"
    ].tolist()

    raise ValueError(
        f"Unsupported raw file format: {invalid}"
    )


print("File-format validation: PASSED")

,dataset_id,file_name,extension,detected_format,supported,size_bytes
0,adult_income,adult_income.csv,.csv,csv,True,3974305
1,bank_marketing,bank_marketing.csv,.csv,csv,True,4610348
2,diabetes_130us,diabetes_130us.csv,.csv,csv,True,16246860


File-format validation: PASSED


In [33]:
# ============================================================
# CELL 01.4 — HEADER VALIDATION
# ============================================================
#
# Purpose:
#   Detect whether the raw dataset contains a valid header.
#
#   Adult Income is a known headerless raw dataset in this
#   project and therefore requires schema-based structural
#   reconstruction in Cell 01.7.
# ============================================================

HEADER_ROWS = []


# ------------------------------------------------------------
# Known headerless datasets
# ------------------------------------------------------------

HEADERLESS_DATASETS = {
    "adult_income"
}


def read_first_line(path):

    with open(
        path,
        "r",
        encoding="utf-8-sig",
        errors="replace"
    ) as file:

        return file.readline().rstrip("\r\n")


for dataset_id, path in RAW_FILE_MAP.items():

    first_line = read_first_line(path)

    if dataset_id in HEADERLESS_DATASETS:

        HEADER_ROWS.append(
            {
                "dataset_id":
                    dataset_id,

                "file_name":
                    path.name,

                "header_present":
                    False,

                "header_status":
                    "KNOWN_HEADERLESS_DATASET",

                "header_raw":
                    first_line,

                "header_length":
                    len(first_line)
            }
        )

    else:

        HEADER_ROWS.append(
            {
                "dataset_id":
                    dataset_id,

                "file_name":
                    path.name,

                "header_present":
                    bool(first_line.strip()),

                "header_status":
                    "HEADER_DETECTED"
                    if first_line.strip()
                    else "HEADER_MISSING",

                "header_raw":
                    first_line,

                "header_length":
                    len(first_line)
            }
        )


HEADER_VALIDATION_DF = pd.DataFrame(
    HEADER_ROWS
)

display(
    HEADER_VALIDATION_DF
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

invalid_header_datasets = (
    HEADER_VALIDATION_DF.loc[
        (
            ~HEADER_VALIDATION_DF["header_present"]
        )
        &
        (
            HEADER_VALIDATION_DF["header_status"]
            != "KNOWN_HEADERLESS_DATASET"
        ),
        "dataset_id"
    ].tolist()
)


if invalid_header_datasets:

    raise ValueError(
        "Unexpected missing header detected for: "
        f"{invalid_header_datasets}"
    )


print()
print("Header validation: PASSED")
print(
    "Known headerless datasets are handled by "
    "schema-based structural repair."
)

,dataset_id,file_name,header_present,header_status,header_raw,header_length
0,adult_income,adult_income.csv,False,KNOWN_HEADERLESS_DATASET,"39, State-gov, 77516, Bachelors, 13, Never-mar...",127
1,bank_marketing,bank_marketing.csv,True,HEADER_DETECTED,"""age"";""job"";""marital"";""education"";""default"";""b...",150
2,diabetes_130us,diabetes_130us.csv,True,HEADER_DETECTED,"race,gender,age,weight,admission_type_id,disch...",641



Header validation: PASSED
Known headerless datasets are handled by schema-based structural repair.


In [34]:
# ============================================================
# CELL 01.5 — DELIMITER VALIDATION
# ============================================================

import csv


def detect_delimiter(path):

    sample_lines = []

    with open(
        path,
        "r",
        encoding="utf-8-sig",
        errors="replace"
    ) as file:

        for _ in range(20):

            line = file.readline()

            if not line:
                break

            if line.strip():
                sample_lines.append(line)

    sample = "".join(sample_lines)

    # First use csv.Sniffer
    try:

        dialect = csv.Sniffer().sniff(
            sample,
            delimiters=",;\t|"
        )

        return dialect.delimiter

    except csv.Error:

        # Deterministic fallback
        counts = {
            ",": sample.count(","),
            ";": sample.count(";"),
            "\t": sample.count("\t"),
            "|": sample.count("|")
        }

        delimiter = max(
            counts,
            key=counts.get
        )

        if counts[delimiter] == 0:
            return None

        return delimiter


DELIMITER_ROWS = []

for dataset_id, path in RAW_FILE_MAP.items():

    delimiter = detect_delimiter(path)

    DELIMITER_ROWS.append(
        {
            "dataset_id": dataset_id,
            "file_name": path.name,
            "delimiter": delimiter,
            "delimiter_name": (
                "comma" if delimiter == ","
                else "semicolon" if delimiter == ";"
                else "tab" if delimiter == "\t"
                else "pipe" if delimiter == "|"
                else "unknown"
            )
        }
    )


DELIMITER_VALIDATION_DF = pd.DataFrame(
    DELIMITER_ROWS
)

display(DELIMITER_VALIDATION_DF)


if DELIMITER_VALIDATION_DF[
    "delimiter"
].isna().any():

    invalid = DELIMITER_VALIDATION_DF.loc[
        DELIMITER_VALIDATION_DF["delimiter"].isna(),
        "dataset_id"
    ].tolist()

    raise ValueError(
        f"Unable to determine delimiter for: {invalid}"
    )


print("Delimiter validation: PASSED")

,dataset_id,file_name,delimiter,delimiter_name
0,adult_income,adult_income.csv,",",comma
1,bank_marketing,bank_marketing.csv,;,semicolon
2,diabetes_130us,diabetes_130us.csv,",",comma


Delimiter validation: PASSED


In [35]:
# ============================================================
# CELL 01.6 — COLUMN NAME STANDARDIZATION
# ============================================================
#
# Purpose:
#   Establish canonical column names without modifying
#   observations or performing feature engineering.
#
# Important:
#   Adult Income is headerless. Its first row is DATA,
#   not a header. Therefore its canonical schema is assigned
#   explicitly and must not be interpreted as column-name
#   standardization of the first observation.
# ============================================================


COLUMN_STANDARDIZATION_RECORDS = []


# ------------------------------------------------------------
# Canonical Adult Income schema
# ------------------------------------------------------------

ADULT_INCOME_COLUMNS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]


# ------------------------------------------------------------
# Process datasets
# ------------------------------------------------------------

for dataset_id, path in RAW_FILE_MAP.items():

    # ========================================================
    # Adult Income
    # ========================================================

    if dataset_id == "adult_income":

        for position, column in enumerate(
            ADULT_INCOME_COLUMNS
        ):

            COLUMN_STANDARDIZATION_RECORDS.append(
                {
                    "dataset_id":
                        dataset_id,

                    "original_column":
                        f"<headerless_column_{position}>",

                    "standardized_column":
                        column,

                    "changed":
                        True,

                    "standardization_type":
                        "canonical_schema_assignment",

                    "reason":
                        "Raw dataset is headerless; "
                        "column names assigned from the "
                        "predefined Adult Income schema."
                }
            )

        continue


    # ========================================================
    # Other datasets
    # ========================================================

    # Read only the header for standardization.
    delimiter = (
        DELIMITER_VALIDATION_DF.loc[
            DELIMITER_VALIDATION_DF["dataset_id"]
            == dataset_id,
            "delimiter"
        ].iloc[0]
    )


    header_df = pd.read_csv(
        path,
        sep=delimiter,
        nrows=0,
        encoding="utf-8-sig",
        engine="python"
    )


    original_columns = list(
        header_df.columns
    )


    standardized_columns = [
        standardize_column_name(
            column
        )
        for column in original_columns
    ]


    standardized_columns = (
        make_unique_columns(
            standardized_columns
        )
    )


    for original, standardized in zip(
        original_columns,
        standardized_columns
    ):

        COLUMN_STANDARDIZATION_RECORDS.append(
            {
                "dataset_id":
                    dataset_id,

                "original_column":
                    original,

                "standardized_column":
                    standardized,

                "changed":
                    original != standardized,

                "standardization_type":
                    "name_normalization",

                "reason":
                    "Canonical column-name normalization."
            }
        )


# ------------------------------------------------------------
# Create audit dataframe
# ------------------------------------------------------------

COLUMN_STANDARDIZATION_DF = pd.DataFrame(
    COLUMN_STANDARDIZATION_RECORDS
)


# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

required_columns = [
    "dataset_id",
    "original_column",
    "standardized_column",
    "changed",
    "standardization_type",
    "reason"
]


missing_audit_columns = [
    column
    for column in required_columns
    if column not in COLUMN_STANDARDIZATION_DF.columns
]


if missing_audit_columns:

    raise RuntimeError(
        "Column standardization audit is incomplete: "
        f"{missing_audit_columns}"
    )


if (
    COLUMN_STANDARDIZATION_DF[
        "standardized_column"
    ]
    .isna()
    .any()
):

    raise RuntimeError(
        "Column standardization produced missing "
        "canonical column names."
    )


display(
    COLUMN_STANDARDIZATION_DF
)


print()
print(
    "Column-name standardization rules established."
)

,dataset_id,original_column,standardized_column,changed,standardization_type,reason
0,adult_income,<headerless_column_0>,age,True,canonical_schema_assignment,Raw dataset is headerless; column names assign...
1,adult_income,<headerless_column_1>,workclass,True,canonical_schema_assignment,Raw dataset is headerless; column names assign...
2,adult_income,<headerless_column_2>,fnlwgt,True,canonical_schema_assignment,Raw dataset is headerless; column names assign...
3,adult_income,<headerless_column_3>,education,True,canonical_schema_assignment,Raw dataset is headerless; column names assign...
4,adult_income,<headerless_column_4>,education_num,True,canonical_schema_assignment,Raw dataset is headerless; column names assign...
...,...,...,...,...,...,...
75,diabetes_130us,metformin-rosiglitazone,metformin_rosiglitazone,True,name_normalization,Canonical column-name normalization.
76,diabetes_130us,metformin-pioglitazone,metformin_pioglitazone,True,name_normalization,Canonical column-name normalization.
77,diabetes_130us,change,change,False,name_normalization,Canonical column-name normalization.
78,diabetes_130us,diabetesMed,diabetesMed,False,name_normalization,Canonical column-name normalization.



Column-name standardization rules established.


In [36]:
# ============================================================
# CELL 01.7 — DATASET-SPECIFIC STRUCTURAL REPAIR
# ============================================================
#
# IMPORTANT:
#   Structural repair ONLY.
#
# This cell does NOT:
#   - impute missing values
#   - generate MCAR/MAR missingness
#   - remove statistical outliers
#   - encode categorical variables
#   - scale numerical variables
#   - perform feature engineering
#
# Those operations belong to later AIR-LLM notebooks.
# ============================================================


# ------------------------------------------------------------
# 1. FINAL CANONICAL SCHEMAS
# ------------------------------------------------------------

ADULT_INCOME_COLUMNS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]


def read_raw_dataset(
    dataset_id,
    path
):
    """
    Read raw dataset while preserving the original data.

    Adult Income is headerless and therefore receives its
    known canonical schema during ingestion.
    """

    delimiter = (
        DELIMITER_VALIDATION_DF.loc[
            DELIMITER_VALIDATION_DF["dataset_id"]
            == dataset_id,
            "delimiter"
        ].iloc[0]
    )


    # --------------------------------------------------------
    # Adult Income — HEADERLESS RAW DATASET
    # --------------------------------------------------------

    if dataset_id == "adult_income":

        df = pd.read_csv(
            path,
            sep=delimiter,
            header=None,
            names=ADULT_INCOME_COLUMNS,
            encoding="utf-8-sig",
            engine="python",
            dtype=str,
            keep_default_na=True,
            na_values=[
                "",
                "?",
                "NA",
                "N/A",
                "NULL",
                "null"
            ],
            skipinitialspace=True
        )

        return df


    # --------------------------------------------------------
    # All other datasets
    # --------------------------------------------------------

    df = pd.read_csv(
        path,
        sep=delimiter,
        encoding="utf-8-sig",
        engine="python",
        dtype=str,
        keep_default_na=True,
        na_values=[
            "",
            "?",
            "NA",
            "N/A",
            "NULL",
            "null"
        ],
        skipinitialspace=True
    )

    return df


# ------------------------------------------------------------
# 2. STRUCTURAL REPAIR FUNCTION
# ------------------------------------------------------------

def structural_repair(
    dataset_id,
    df
):

    repair_log = []

    original_shape = df.shape


    # --------------------------------------------------------
    # Adult Income
    # --------------------------------------------------------

    if dataset_id == "adult_income":

        # Ensure exact canonical schema
        if df.shape[1] != len(
            ADULT_INCOME_COLUMNS
        ):

            raise ValueError(
                "Adult Income schema mismatch.\n"
                f"Expected columns: "
                f"{len(ADULT_INCOME_COLUMNS)}\n"
                f"Detected columns: "
                f"{df.shape[1]}"
            )

        old_columns = list(df.columns)

        df.columns = (
            ADULT_INCOME_COLUMNS
        )

        if old_columns != list(
            ADULT_INCOME_COLUMNS
        ):

            repair_log.append(
                {
                    "dataset_id":
                        dataset_id,

                    "repair":
                        "adult_income_schema_assignment",

                    "details":
                        "Assigned canonical 15-column "
                        "headerless Adult Income schema."
                }
            )


        # ----------------------------------------------------
        # Normalize structural whitespace only.
        #
        # This does NOT alter the semantic values.
        # ----------------------------------------------------

        for column in df.columns:

            if (
                df[column].dtype
                == "object"
            ):

                df[column] = (
                    df[column]
                    .str.strip()
                )


        repair_log.append(
            {
                "dataset_id":
                    dataset_id,

                "repair":
                    "adult_income_whitespace_normalization",

                "details":
                    "Removed leading/trailing whitespace."
            }
        )


    # --------------------------------------------------------
    # Bank Marketing
    # --------------------------------------------------------

    elif dataset_id == "bank_marketing":

        original_columns = list(
            df.columns
        )

        standardized_columns = [
            standardize_column_name(
                column
            )
            for column in df.columns
        ]

        standardized_columns = (
            make_unique_columns(
                standardized_columns
            )
        )

        df.columns = (
            standardized_columns
        )

        if (
            original_columns
            != standardized_columns
        ):

            repair_log.append(
                {
                    "dataset_id":
                        dataset_id,

                    "repair":
                        "bank_marketing_column_standardization",

                    "details":
                        str(
                            list(
                                zip(
                                    original_columns,
                                    standardized_columns
                                )
                            )
                        )
                }
            )


        # Target normalization
        target_aliases = {
            "target": "y",
            "response": "y",
            "deposit": "y"
        }

        rename_map = {}

        for column in df.columns:

            normalized = (
                str(column)
                .strip()
                .lower()
            )

            if normalized in target_aliases:

                target_name = (
                    target_aliases[
                        normalized
                    ]
                )

                if (
                    target_name
                    not in df.columns
                ):

                    rename_map[
                        column
                    ] = target_name


        if rename_map:

            df = df.rename(
                columns=rename_map
            )

            repair_log.append(
                {
                    "dataset_id":
                        dataset_id,

                    "repair":
                        "bank_marketing_target_standardization",

                    "details":
                        str(rename_map)
                }
            )


    # --------------------------------------------------------
    # Diabetes 130-US
    # --------------------------------------------------------

    elif dataset_id == "diabetes_130us":

        original_columns = list(
            df.columns
        )

        standardized_columns = [
            standardize_column_name(
                column
            )
            for column in df.columns
        ]

        standardized_columns = (
            make_unique_columns(
                standardized_columns
            )
        )

        df.columns = (
            standardized_columns
        )

        if (
            original_columns
            != standardized_columns
        ):

            repair_log.append(
                {
                    "dataset_id":
                        dataset_id,

                    "repair":
                        "diabetes_column_standardization",

                    "details":
                        str(
                            list(
                                zip(
                                    original_columns,
                                    standardized_columns
                                )
                            )
                        )
                }
            )


        # Readmission target aliases
        readmission_aliases = [
            "readmission",
            "readmission_status"
        ]

        rename_map = {}

        for column in df.columns:

            if (
                str(column).lower()
                in readmission_aliases
            ):

                if (
                    "readmitted"
                    not in df.columns
                ):

                    rename_map[
                        column
                    ] = "readmitted"


        if rename_map:

            df = df.rename(
                columns=rename_map
            )

            repair_log.append(
                {
                    "dataset_id":
                        dataset_id,

                    "repair":
                        "diabetes_target_standardization",

                    "details":
                        str(rename_map)
                }
            )


    # --------------------------------------------------------
    # Common structural cleanup
    # --------------------------------------------------------

    # Remove completely empty columns only.
    #
    # No meaningful information is removed.

    empty_columns = [
        column
        for column in df.columns
        if df[column].isna().all()
    ]

    if empty_columns:

        df = df.drop(
            columns=empty_columns
        )

        repair_log.append(
            {
                "dataset_id":
                    dataset_id,

                "repair":
                    "remove_completely_empty_columns",

                "details":
                    str(empty_columns)
            }
        )


    # --------------------------------------------------------
    # Ensure unique column names
    # --------------------------------------------------------

    if not df.columns.is_unique:

        old_columns = list(
            df.columns
        )

        df.columns = (
            make_unique_columns(
                df.columns
            )
        )

        repair_log.append(
            {
                "dataset_id":
                    dataset_id,

                "repair":
                    "duplicate_column_name_resolution",

                "details":
                    str(
                        list(
                            zip(
                                old_columns,
                                df.columns
                            )
                        )
                    )
            }
        )


    # --------------------------------------------------------
    # Final structural summary
    # --------------------------------------------------------

    repair_summary = {

        "dataset_id":
            dataset_id,

        "rows_before":
            int(original_shape[0]),

        "columns_before":
            int(original_shape[1]),

        "rows_after":
            int(df.shape[0]),

        "columns_after":
            int(df.shape[1]),

        "repair_count":
            int(len(repair_log))
    }


    return (
        df,
        repair_log,
        repair_summary
    )


# ------------------------------------------------------------
# 3. PROCESS ALL DATASETS
# ------------------------------------------------------------

RAW_DATASETS = {}

STRUCTURAL_REPAIR_LOG = []

STRUCTURAL_REPAIR_SUMMARY = []


for dataset_id, path in RAW_FILE_MAP.items():

    raw_df = read_raw_dataset(
        dataset_id,
        path
    )

    repaired_df, repair_log, summary = (
        structural_repair(
            dataset_id,
            raw_df
        )
    )

    RAW_DATASETS[
        dataset_id
    ] = repaired_df

    STRUCTURAL_REPAIR_LOG.extend(
        repair_log
    )

    STRUCTURAL_REPAIR_SUMMARY.append(
        summary
    )


STRUCTURAL_REPAIR_SUMMARY_DF = (
    pd.DataFrame(
        STRUCTURAL_REPAIR_SUMMARY
    )
)


display(
    STRUCTURAL_REPAIR_SUMMARY_DF
)


# ------------------------------------------------------------
# 4. Adult Income verification
# ------------------------------------------------------------

adult_df = RAW_DATASETS[
    "adult_income"
]


print()
print("=" * 100)
print("ADULT INCOME — CANONICAL STRUCTURAL VERIFICATION")
print("=" * 100)

print(
    f"Shape : {adult_df.shape}"
)

print()
print("Columns:")

for index, column in enumerate(
    adult_df.columns
):

    print(
        f"{index:2d} : {column}"
    )

print()
print("Target distribution:")

print(
    adult_df[
        "income"
    ]
    .value_counts(
        dropna=False
    )
)

print()
print(
    "Adult Income target column:",
    "income"
    if "income" in adult_df.columns
    else "MISSING"
)


if (
    "income"
    not in adult_df.columns
):

    raise RuntimeError(
        "Adult Income structural repair failed: "
        "canonical target 'income' was not created."
    )


print()
print(
    "Adult Income structural repair: PASSED"
)

,dataset_id,rows_before,columns_before,rows_after,columns_after,repair_count
0,adult_income,32561,15,32561,15,1
1,bank_marketing,45211,17,45211,17,0
2,diabetes_130us,101766,48,101766,48,1



ADULT INCOME — CANONICAL STRUCTURAL VERIFICATION
Shape : (32561, 15)

Columns:
 0 : age
 1 : workclass
 2 : fnlwgt
 3 : education
 4 : education_num
 5 : marital_status
 6 : occupation
 7 : relationship
 8 : race
 9 : sex
10 : capital_gain
11 : capital_loss
12 : hours_per_week
13 : native_country
14 : income

Target distribution:
income
<=50K    24720
>50K      7841
Name: count, dtype: int64

Adult Income target column: income

Adult Income structural repair: PASSED


In [37]:
# ============================================================
# CELL 01.8 — DUPLICATE DETECTION
# ============================================================

DUPLICATE_ROWS = []

for dataset_id, df in RAW_DATASETS.items():

    exact_duplicate_count = int(
        df.duplicated(
            keep=False
        ).sum()
    )

    duplicate_row_count = int(
        df.duplicated(
            keep="first"
        ).sum()
    )

    unique_row_count = int(
        df.drop_duplicates().shape[0]
    )

    D = {

        "dataset_id":
            dataset_id,

        "total_rows":
            int(len(df)),

        "unique_rows":
            unique_row_count,

        "duplicate_rows_after_first":
            duplicate_row_count,

        "duplicate_rows_including_first":
            exact_duplicate_count,

        "duplicate_percentage":
            (
                duplicate_row_count
                / len(df)
                * 100
                if len(df) > 0
                else 0.0
            )
    }

    DUPLICATE_ROWS.append(D)


DUPLICATE_DF = pd.DataFrame(
    DUPLICATE_ROWS
)

display(DUPLICATE_DF)


print(
    "Duplicate detection completed."
)

print(
    "No duplicate rows are automatically removed. "
    "The raw dataset remains preserved."
)

,dataset_id,total_rows,unique_rows,duplicate_rows_after_first,duplicate_rows_including_first,duplicate_percentage
0,adult_income,32561,32537,24,47,0.073708
1,bank_marketing,45211,45211,0,0,0.000000
2,diabetes_130us,101766,101766,0,0,0.000000


Duplicate detection completed.
No duplicate rows are automatically removed. The raw dataset remains preserved.


In [38]:
# ============================================================
# CELL 01.9 — RAW DATASET STATISTICS
# ============================================================

RAW_STATISTICS_ROWS = []

for dataset_id, df in RAW_DATASETS.items():

    target = None

    dataset_config = DATASETS_CONFIG[
        dataset_id
    ]

    if isinstance(
        dataset_config,
        dict
    ):

        target = registry_value(
            dataset_config,
            [
                "target",
                "target_column",
                "target_feature"
            ]
        )

    # Fallback to finalized AIR-LLM targets
    if dataset_id == "adult_income":
        target = "income"

    elif dataset_id == "bank_marketing":
        target = "y"

    elif dataset_id == "diabetes_130us":
        target = "readmitted"


    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_columns = [
        column
        for column in df.columns
        if column not in numeric_columns
    ]

    missing_cells = int(
        df.isna().sum().sum()
    )

    total_cells = int(
        df.shape[0] * df.shape[1]
    )

    RAW_STATISTICS_ROWS.append(
        {
            "dataset_id":
                dataset_id,

            "rows":
                int(df.shape[0]),

            "columns":
                int(df.shape[1]),

            "numeric_columns":
                len(numeric_columns),

            "categorical_columns":
                len(categorical_columns),

            "missing_cells":
                missing_cells,

            "missing_cell_percentage":
                (
                    missing_cells
                    / total_cells
                    * 100
                    if total_cells > 0
                    else 0.0
                ),

            "duplicate_rows":
                int(
                    df.duplicated().sum()
                ),

            "target":
                target,

            "target_exists":
                target in df.columns
                if target
                else False
        }
    )


RAW_STATISTICS_DF = pd.DataFrame(
    RAW_STATISTICS_ROWS
)

display(RAW_STATISTICS_DF)


# ------------------------------------------------------------
# Column-level structural statistics
# ------------------------------------------------------------

COLUMN_STATISTICS_ROWS = []

for dataset_id, df in RAW_DATASETS.items():

    for column in df.columns:

        COLUMN_STATISTICS_ROWS.append(
            {
                "dataset_id":
                    dataset_id,

                "column":
                    column,

                "dtype":
                    str(df[column].dtype),

                "non_null_count":
                    int(
                        df[column].notna().sum()
                    ),

                "missing_count":
                    int(
                        df[column].isna().sum()
                    ),

                "missing_percentage":
                    float(
                        df[column].isna().mean()
                        * 100
                    ),

                "unique_count":
                    int(
                        df[column].nunique(
                            dropna=True
                        )
                    )
            }
        )


COLUMN_STATISTICS_DF = pd.DataFrame(
    COLUMN_STATISTICS_ROWS
)

display(
    COLUMN_STATISTICS_DF.head(30)
)

,dataset_id,rows,columns,numeric_columns,categorical_columns,missing_cells,missing_cell_percentage,duplicate_rows,target,target_exists
0,adult_income,32561,15,0,15,4262,0.872619,24,income,True
1,bank_marketing,45211,17,0,17,0,0.000000,0,y,True
2,diabetes_130us,101766,48,0,48,374017,7.656802,0,readmitted,True


,dataset_id,column,dtype,non_null_count,missing_count,missing_percentage,unique_count
0,adult_income,age,object,32561,0,0.000000,73
1,adult_income,workclass,object,30725,1836,5.638647,8
2,adult_income,fnlwgt,object,32561,0,0.000000,21648
3,adult_income,education,object,32561,0,0.000000,16
4,adult_income,education_num,object,32561,0,0.000000,16
5,adult_income,marital_status,object,32561,0,0.000000,7
6,adult_income,occupation,object,30718,1843,5.660146,14
7,adult_income,relationship,object,32561,0,0.000000,6
8,adult_income,race,object,32561,0,0.000000,5
9,adult_income,sex,object,32561,0,0.000000,2


In [39]:
# ============================================================
# CELL 01.10 — CANONICAL DATASET EXPORT
# ============================================================
#
# Canonical datasets are the structurally validated versions
# that will be consumed by Notebook 02.
#
# Raw files are NEVER overwritten.
# ============================================================

CANONICAL_DATASETS = {}

CANONICAL_PATHS = {}

for dataset_id, df in RAW_DATASETS.items():

    canonical_df = df.copy()

    # --------------------------------------------------------
    # Preserve column order established during structural
    # validation.
    # --------------------------------------------------------

    canonical_df = canonical_df.loc[
        :,
        list(canonical_df.columns)
    ]

    canonical_path = (
        PROCESSED_DIR
        / f"{dataset_id}_canonical.csv"
    )

    canonical_df.to_csv(
        canonical_path,
        index=False,
        encoding="utf-8"
    )

    CANONICAL_DATASETS[
        dataset_id
    ] = canonical_df

    CANONICAL_PATHS[
        dataset_id
    ] = canonical_path


CANONICAL_EXPORT_ROWS = []

for dataset_id, path in CANONICAL_PATHS.items():

    df = CANONICAL_DATASETS[
        dataset_id
    ]

    CANONICAL_EXPORT_ROWS.append(
        {
            "dataset_id":
                dataset_id,

            "canonical_file":
                str(path),

            "rows":
                int(df.shape[0]),

            "columns":
                int(df.shape[1]),

            "file_size_bytes":
                int(path.stat().st_size)
        }
    )


CANONICAL_EXPORT_DF = pd.DataFrame(
    CANONICAL_EXPORT_ROWS
)

display(
    CANONICAL_EXPORT_DF
)


# ------------------------------------------------------------
# Save structural repair log
# ------------------------------------------------------------

REPAIR_LOG_DF = pd.DataFrame(
    STRUCTURAL_REPAIR_LOG
)

REPAIR_LOG_PATH = (
    REFERENCE_DIR
    / "dataset_structural_repair_log.csv"
)

if REPAIR_LOG_DF.empty:

    REPAIR_LOG_DF = pd.DataFrame(
        columns=[
            "dataset_id",
            "repair",
            "details"
        ]
    )

REPAIR_LOG_DF.to_csv(
    REPAIR_LOG_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# Save duplicate report
# ------------------------------------------------------------

DUPLICATE_REPORT_PATH = (
    REFERENCE_DIR
    / "raw_duplicate_detection.csv"
)

DUPLICATE_DF.to_csv(
    DUPLICATE_REPORT_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# Save raw statistics
# ------------------------------------------------------------

RAW_STATS_PATH = (
    REFERENCE_DIR
    / "raw_dataset_statistics.csv"
)

RAW_STATISTICS_DF.to_csv(
    RAW_STATS_PATH,
    index=False,
    encoding="utf-8"
)


COLUMN_STATS_PATH = (
    REFERENCE_DIR
    / "raw_column_statistics.csv"
)

COLUMN_STATISTICS_DF.to_csv(
    COLUMN_STATS_PATH,
    index=False,
    encoding="utf-8"
)


print("=" * 100)
print("CANONICAL DATASET EXPORT COMPLETED")
print("=" * 100)

for dataset_id, path in CANONICAL_PATHS.items():

    print(
        f"{dataset_id:20s} -> {path}"
    )

print()
print(
    f"Repair log       : {REPAIR_LOG_PATH}"
)

print(
    f"Duplicate report : {DUPLICATE_REPORT_PATH}"
)

print(
    f"Dataset stats    : {RAW_STATS_PATH}"
)

print(
    f"Column stats     : {COLUMN_STATS_PATH}"
)

,dataset_id,canonical_file,rows,columns,file_size_bytes
0,adult_income,/content/drive/MyDrive/AIR_LLM_Research/data/p...,32561,15,3514344
1,bank_marketing,/content/drive/MyDrive/AIR_LLM_Research/data/p...,45211,17,3706094
2,diabetes_130us,/content/drive/MyDrive/AIR_LLM_Research/data/p...,101766,48,16246860


CANONICAL DATASET EXPORT COMPLETED
adult_income         -> /content/drive/MyDrive/AIR_LLM_Research/data/processed/adult_income_canonical.csv
bank_marketing       -> /content/drive/MyDrive/AIR_LLM_Research/data/processed/bank_marketing_canonical.csv
diabetes_130us       -> /content/drive/MyDrive/AIR_LLM_Research/data/processed/diabetes_130us_canonical.csv

Repair log       : /content/drive/MyDrive/AIR_LLM_Research/data/reference/dataset_structural_repair_log.csv
Duplicate report : /content/drive/MyDrive/AIR_LLM_Research/data/reference/raw_duplicate_detection.csv
Dataset stats    : /content/drive/MyDrive/AIR_LLM_Research/data/reference/raw_dataset_statistics.csv
Column stats     : /content/drive/MyDrive/AIR_LLM_Research/data/reference/raw_column_statistics.csv


In [40]:
# ============================================================
# CELL 01.11 — DATASET INTEGRITY VALIDATION
# ============================================================
#
# Final gate for Notebook 01.
#
# Notebook 02 should only be started when every required
# integrity check passes.
# ============================================================

INTEGRITY_ROWS = []


# ------------------------------------------------------------
# Expected final targets
# ------------------------------------------------------------

EXPECTED_TARGETS = {

    "adult_income":
        "income",

    "bank_marketing":
        "y",

    "diabetes_130us":
        "readmitted"
}


# ------------------------------------------------------------
# Validate each canonical dataset
# ------------------------------------------------------------

for dataset_id in EXPECTED_DATASETS:

    checks = {}

    canonical_path = (
        CANONICAL_PATHS.get(
            dataset_id
        )
    )

    canonical_df = (
        CANONICAL_DATASETS.get(
            dataset_id
        )
    )

    raw_path = (
        RAW_FILE_MAP.get(
            dataset_id
        )
    )


    # File existence
    checks["raw_file_exists"] = (
        raw_path is not None
        and raw_path.exists()
    )

    checks["canonical_file_exists"] = (
        canonical_path is not None
        and canonical_path.exists()
    )


    # Dataset object
    checks["dataset_loaded"] = (
        canonical_df is not None
    )


    if canonical_df is not None:

        # Non-empty
        checks["non_empty"] = (
            canonical_df.shape[0] > 0
            and canonical_df.shape[1] > 0
        )

        # Unique columns
        checks["unique_column_names"] = (
            canonical_df.columns.is_unique
        )

        # No empty column names
        checks["non_empty_column_names"] = all(
            str(column).strip() != ""
            for column in canonical_df.columns
        )

        # Target
        expected_target = (
            EXPECTED_TARGETS[
                dataset_id
            ]
        )

        checks["target_exists"] = (
            expected_target
            in canonical_df.columns
        )

        # Row count
        checks["row_count_preserved"] = (
            canonical_df.shape[0]
            ==
            RAW_DATASETS[
                dataset_id
            ].shape[0]
        )

        # Column count
        checks["column_count_valid"] = (
            canonical_df.shape[1] > 0
        )

        # No duplicate column names
        checks["duplicate_columns_absent"] = (
            len(
                canonical_df.columns
            )
            ==
            len(
                set(
                    canonical_df.columns
                )
            )
        )

        # No index export
        checks["index_column_absent"] = not any(
            str(column).lower().startswith(
                "unnamed"
            )
            for column in canonical_df.columns
        )

    else:

        checks["non_empty"] = False
        checks["unique_column_names"] = False
        checks["non_empty_column_names"] = False
        checks["target_exists"] = False
        checks["row_count_preserved"] = False
        checks["column_count_valid"] = False
        checks["duplicate_columns_absent"] = False
        checks["index_column_absent"] = False


    checks["all_checks_passed"] = all(
        checks.values()
    )


    INTEGRITY_ROWS.append(
        {
            "dataset_id":
                dataset_id,

            **checks
        }
    )


INTEGRITY_DF = pd.DataFrame(
    INTEGRITY_ROWS
)

display(
    INTEGRITY_DF
)


# ------------------------------------------------------------
# Global validation
# ------------------------------------------------------------

CHECK_COLUMNS = [
    column
    for column in INTEGRITY_DF.columns
    if column not in {
        "dataset_id",
        "all_checks_passed"
    }
]

ALL_INTEGRITY_PASSED = bool(
    INTEGRITY_DF[
        "all_checks_passed"
    ].all()
)


# ------------------------------------------------------------
# Save validation report
# ------------------------------------------------------------

INTEGRITY_REPORT_PATH = (
    REFERENCE_DIR
    / "dataset_integrity_validation.csv"
)

INTEGRITY_DF.to_csv(
    INTEGRITY_REPORT_PATH,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------
# Save Notebook 01 manifest
# ------------------------------------------------------------

NOTEBOOK_01_MANIFEST = {

    "notebook":
        "01_Dataset_Acquisition_and_Validation",

    "project":
        "AIR-LLM Research Project",

    "project_code":
        "AIR-LLM",

    "project_version":
        "1.0",

    "experiment_version":
        "AIR-LLM-v1",

    "configuration_version":
        "CONFIG-v1",

    "datasets":
        EXPECTED_DATASETS,

    "targets":
        EXPECTED_TARGETS,

    "raw_directory":
        str(RAW_DIR),

    "canonical_directory":
        str(PROCESSED_DIR),

    "dataset_count":
        len(EXPECTED_DATASETS),

    "integrity_validation":
        "PASSED"
        if ALL_INTEGRITY_PASSED
        else "FAILED",

    "integrity_report":
        str(INTEGRITY_REPORT_PATH),

    "structural_repair_log":
        str(REPAIR_LOG_PATH),

    "duplicate_report":
        str(DUPLICATE_REPORT_PATH),

    "raw_statistics":
        str(RAW_STATS_PATH),

    "column_statistics":
        str(COLUMN_STATS_PATH),

    "canonical_datasets": {
        dataset_id:
            str(path)
        for dataset_id, path
        in CANONICAL_PATHS.items()
    }
}


NOTEBOOK_01_MANIFEST_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "notebook_01_manifest.json"
)

NOTEBOOK_01_MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    NOTEBOOK_01_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        NOTEBOOK_01_MANIFEST,
        file,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# Final gate
# ------------------------------------------------------------

if not ALL_INTEGRITY_PASSED:

    failed_datasets = INTEGRITY_DF.loc[
        ~INTEGRITY_DF["all_checks_passed"],
        "dataset_id"
    ].tolist()

    raise RuntimeError(
        "NOTEBOOK 01 INTEGRITY VALIDATION FAILED.\n"
        f"Failed datasets: {failed_datasets}\n"
        "Do not continue to Notebook 02 until resolved."
    )


print()
print("=" * 100)
print("AIR-LLM — NOTEBOOK 01 COMPLETE")
print("=" * 100)

print()
print("DATASETS")
print("-" * 100)

for dataset_id in EXPECTED_DATASETS:

    df = CANONICAL_DATASETS[
        dataset_id
    ]

    print(
        f"{dataset_id:20s} | "
        f"Rows: {df.shape[0]:,} | "
        f"Columns: {df.shape[1]:,} | "
        f"Target: {EXPECTED_TARGETS[dataset_id]}"
    )


print()
print("VALIDATION")
print("-" * 100)

print(
    f"Datasets validated       : "
    f"{len(EXPECTED_DATASETS)}"
)

print(
    f"Raw files detected       : "
    f"{len(RAW_FILE_MAP)}"
)

print(
    f"Canonical datasets       : "
    f"{len(CANONICAL_PATHS)}"
)

print(
    f"Integrity validation     : "
    f"{'PASSED' if ALL_INTEGRITY_PASSED else 'FAILED'}"
)

print()
print("OUTPUTS")
print("-" * 100)

print(
    f"Canonical datasets       : {PROCESSED_DIR}"
)

print(
    f"Integrity report         : "
    f"{INTEGRITY_REPORT_PATH}"
)

print(
    f"Structural repair log    : "
    f"{REPAIR_LOG_PATH}"
)

print(
    f"Duplicate report         : "
    f"{DUPLICATE_REPORT_PATH}"
)

print(
    f"Raw statistics           : "
    f"{RAW_STATS_PATH}"
)

print(
    f"Column statistics        : "
    f"{COLUMN_STATS_PATH}"
)

print(
    f"Notebook manifest        : "
    f"{NOTEBOOK_01_MANIFEST_PATH}"
)

print()
print("=" * 100)
print("ALL NOTEBOOK 01 VALIDATIONS PASSED")
print("CANONICAL DATASETS ARE READY FOR NOTEBOOK 02")
print("=" * 100)

,dataset_id,raw_file_exists,canonical_file_exists,dataset_loaded,non_empty,unique_column_names,non_empty_column_names,target_exists,row_count_preserved,column_count_valid,duplicate_columns_absent,index_column_absent,all_checks_passed
0,adult_income,True,True,True,True,True,True,True,True,True,True,True,True
1,bank_marketing,True,True,True,True,True,True,True,True,True,True,True,True
2,diabetes_130us,True,True,True,True,True,True,True,True,True,True,True,True



AIR-LLM — NOTEBOOK 01 COMPLETE

DATASETS
----------------------------------------------------------------------------------------------------
adult_income         | Rows: 32,561 | Columns: 15 | Target: income
bank_marketing       | Rows: 45,211 | Columns: 17 | Target: y
diabetes_130us       | Rows: 101,766 | Columns: 48 | Target: readmitted

VALIDATION
----------------------------------------------------------------------------------------------------
Datasets validated       : 3
Raw files detected       : 3
Canonical datasets       : 3
Integrity validation     : PASSED

OUTPUTS
----------------------------------------------------------------------------------------------------
Canonical datasets       : /content/drive/MyDrive/AIR_LLM_Research/data/processed
Integrity report         : /content/drive/MyDrive/AIR_LLM_Research/data/reference/dataset_integrity_validation.csv
Structural repair log    : /content/drive/MyDrive/AIR_LLM_Research/data/reference/dataset_structural_repair_log.csv